In [1]:
!pip install -q fastparquet
!pip install -q statsforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 50.3 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 47.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 3.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import TSB, AutoETS, AutoARIMA, Naive, SeasonalNaive, Theta, OptimizedTheta, CrostonOptimized, ADIDA, IMAPA, CrostonSBA, HoltWinters

%matplotlib inline

#####################################################
# Импортируем .py файлы с уже написанными функциями #
#####################################################

import sys
sys.path.append('/kaggle/input/datasets/faibus/diploma')

# функции для расчёта метрик
from metrics import (
    DEFAULT_METRICS,
    get_model_columns,
    compute_metrics_per_window,
    summarize_metrics,
    summarize_metrics_by_segments,
)

# функции для фильтрации и разделения рядов
from split_utils import filter_long_series, split_train_val_test

# функции для оценки и сбора предсказаний
from evaluation_utils import evaluate_frozen_windows

In [3]:
# данные о реальном спросе
real_demand = pd.read_parquet('/kaggle/input/datasets/faibus/diploma/real_demand_data.parquet', engine='fastparquet')

# выгружаем типы рядов
ts_dict_df = pd.read_csv('/kaggle/input/datasets/faibus/diploma/ts_dict_df')[['SKU_id', 'ts_type']]

In [4]:
# Выделяем SKU, принадлежащие к типу lumpy
lumpy_ts = list(ts_dict_df.query(" ts_type == 'erratic' ")['SKU_id'])

# фильтруем датасет по lumpy_ts
real_demand = real_demand.query(" SKU_id.isin(@lumpy_ts) ")

# причесываем датасет
real_demand = real_demand.rename(columns = {'date':'ds', 'real_demand':'y', 'SKU_id':'unique_id'})[['unique_id', 'ds', 'y']]

real_demand.shape

(298461, 3)

### Делим на train / val / test

In [5]:
from split_utils import filter_long_series, split_train_val_test

real_demand_filtered = filter_long_series(
    real_demand,
    horizon=14,
    n_val_windows=1,
    n_test_windows=3,
    min_train_points=35,
    id_col="unique_id",
)

eval_df = real_demand_filtered[["unique_id", "ds", "y"]].sort_values(["unique_id", "ds"])

### Обучение

In [7]:
models = [
    SeasonalNaive(season_length=7, alias='SeasonalNaive7'),
    Naive(alias='Naive'),
    AutoETS(alias='AutoETS'),
    AutoARIMA(alias='AutoARIMA'),
    OptimizedTheta(alias='OptTheta'),
    ADIDA(alias='ADIDA'),
    IMAPA(alias='IMAPA')
]

# обучаем только на train, val - подбор наилучших параметров, test - итоговое качество метрик
sf = StatsForecast(models=models, freq="D", n_jobs=1)
cv_frozen = sf.cross_validation(
    df=eval_df,
    h=14,
    step_size=14,
    n_windows=4,   # 1 val + 3 test
    refit=False,   # не переобучаем параметры
)

### Собираем прогнозы

In [13]:
# разрезаем на val_pred и test_pred
cutoff_stats = (
    cv_frozen.groupby("cutoff")["unique_id"]
    .nunique()
    .rename("n_ids")
    .reset_index()
    .sort_values("cutoff")
)

max_ids = cutoff_stats["n_ids"].max()

# Берем только "полные" окна (с максимальным числом рядов)
full_cutoffs = cutoff_stats.loc[cutoff_stats["n_ids"] == max_ids, "cutoff"].tolist()

# Нужны последние 4 окна: 1 val + 3 test
selected_cutoffs = full_cutoffs[-4:]
val_cutoff = selected_cutoffs[0]
test_cutoffs = selected_cutoffs[1:]

val_pred = cv_frozen[cv_frozen["cutoff"] == val_cutoff].copy()
test_pred = cv_frozen[cv_frozen["cutoff"].isin(test_cutoffs)].copy()

cutoff_to_window = {c: i + 1 for i, c in enumerate(test_cutoffs)}
test_pred["test_window"] = test_pred["cutoff"].map(cutoff_to_window)

### Считаем метрики

In [14]:
from metrics import DEFAULT_METRICS, get_model_columns, compute_metrics_per_window, summarize_metrics

val_model_cols = get_model_columns(
    val_pred,
    reserved_columns=("unique_id", "ds", "cutoff", "y", "index", "test_window"),
)
val_metrics_per_window = compute_metrics_per_window(val_pred, val_model_cols, DEFAULT_METRICS)
val_summary_mean, val_summary_stats = summarize_metrics(val_metrics_per_window)

test_model_cols = get_model_columns(
    test_pred,
    reserved_columns=("unique_id", "ds", "cutoff", "y", "index", "test_window"),
)
test_metrics_per_window = compute_metrics_per_window(test_pred, test_model_cols, DEFAULT_METRICS)
test_summary_mean, test_summary_stats = summarize_metrics(test_metrics_per_window)

print("VAL mean metrics:")
display(val_summary_mean)

print('')

print("TEST mean metrics (3 windows x 14 days):")
display(test_summary_mean)

VAL mean metrics:


,mae,rmse,smape,wape
model,,,,
ADIDA,5.328042,11.975335,78.396495,66.040417
AutoARIMA,5.482040,11.938222,78.554296,67.949196
AutoETS,5.743221,13.927130,79.077470,71.186512
IMAPA,5.333546,11.966674,78.451052,66.108631
Naive,6.240392,14.384289,89.799525,77.348883
OptTheta,5.485627,12.729040,78.857483,67.993665
SeasonalNaive7,6.304797,14.182895,94.839527,78.147175



TEST mean metrics (3 windows x 14 days):


,mae,rmse,smape,wape
model,,,,
ADIDA,6.283328,27.067650,78.682838,71.526426
AutoARIMA,6.372729,27.607718,78.516470,72.640902
AutoETS,6.675418,28.349351,80.832225,76.032219
IMAPA,6.286523,27.078871,78.806794,71.564469
Naive,6.903083,27.939519,91.831486,78.678502
OptTheta,6.554618,27.629446,81.067325,74.669828
SeasonalNaive7,7.434667,30.280673,95.082959,84.683418


In [16]:
test_pred.to_csv("erratic_forecast.csv")